*Nombre de Grupos*:

*Integrantes*:

# **Laboratorio 6: SIFT y Bag of Words**

Vamos a utilizar un descriptor SIFT para gnenerar un espacio latente que nos sirva para encontrar fotos similares a una referencia.

> **Parte 1: SIFT.** Cómo encuentra SIFT puntos que se pueden reconocer entre distintas fotos, aunque cambie el tamaño o la rotación, y cómo los describe.
\
> **Parte 2: Bag of Words.** Con esos descriptores armamos un "vocabulario visual" y convertimos cada imagen en un vector de largo fijo. Después buscamos imágenes parecidas y medimos qué tan bien funciona.
\
> **Parte 3: Autoencoder.** Entrenamos una red que comprime esos vectores y vemos si el nuevo espacio ordena mejor las imágenes.

## **Librerías**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import LineCollection, PatchCollection
from matplotlib.patches import Circle, Polygon
from matplotlib.lines import Line2D
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox, Dropdown

from skimage import data, color, filters, feature, transform, measure

SEMILLA = 11

## **Estilo de los gráficos**

In [ ]:
#@title Paleta y estilo { display-mode: "form" }

# Azul y naranja son los mismos de las clases de MLP. El aqua es la tercera clase de la parte 2
COLOR_CLASE_0 = "#2B6CB0"
COLOR_CLASE_1 = "#E8871E"
COLOR_CLASE_2 = "#1BAF7A"
COLOR_TINTA = "#1F2933"

COLORES_CLASES = [COLOR_CLASE_0, COLOR_CLASE_1, COLOR_CLASE_2]
# La forma repite lo que dice el color, por si se proyecta lavado o se imprime en blanco y negro
MARCADORES_CLASES = ["o", "s", "^"]

MAPA_DIVERGENTE = LinearSegmentedColormap.from_list("divergente", [COLOR_CLASE_0, "#F6F4F1", COLOR_CLASE_1])
MAPA_SECUENCIAL = LinearSegmentedColormap.from_list("secuencial", ["#F6F4F1", COLOR_CLASE_0])

plt.rcParams.update({
  "figure.facecolor": "white",
  "axes.facecolor": "white",
  "axes.edgecolor": COLOR_TINTA,
  "axes.labelcolor": COLOR_TINTA,
  "axes.titlelocation": "left",
  "axes.titleweight": "bold",
  "axes.titlesize": 12,
  "axes.spines.top": False,
  "axes.spines.right": False,
  "text.color": COLOR_TINTA,
  "xtick.color": COLOR_TINTA,
  "ytick.color": COLOR_TINTA,
  "font.size": 11,
})

# **Parte 1: SIFT**

**SIFT** (*Scale-Invariant Feature Transform*) encuentra puntos de una imagen que se pueden volver a encontrar en otra foto de la misma escena, aunque la foto esté más cerca, más lejos o rotada. A cada punto le calcula un **descriptor**: un vector de 128 números que lo describe.

Lo hace en cuatro pasos:

> 1. **Espacio de escalas:** suaviza la imagen con $\sigma$ cada vez más grande y resta imágenes consecutivas (*Difference of Gaussians*).

> 2. **Keypoints:** se queda con los máximos y mínimos de esas diferencias, tanto en la imagen como entre escalas, y descarta los de poco contraste y los que están sobre bordes.

> 3. **Orientación:** mide hacia dónde apuntan los gradientes alrededor del punto y toma la dirección dominante.

> 4. **Descriptor:** gira una ventana según esa orientación, la divide en 4x4 celdas y en cada una arma un histograma de 8 orientaciones de gradiente: $4 \times 4 \times 8 = 128$ números. Es la misma idea que **HOG**.

## **Difference of Gaussians (DoG)**

Una **pirámide Gaussiana** es la misma imagen suavizada con un $\sigma$ cada vez mayor: los detalles finos desaparecen primero y quedan las estructuras grandes. Restando dos niveles consecutivos nos quedamos justo con lo que hay entre una escala y la siguiente:

$$
D_i(x, y) = G_{\sigma_{i+1}} * I - G_{\sigma_i} * I
$$

La DoG aproxima al Laplaciano de la gaussiana y responde a las **manchas** del tamaño de $\sigma$. Es positiva donde la imagen más borrosa es más clara que la nítida, y negativa donde es más oscura.

### ***Ejercicio***: Pirámide Gaussiana y DoG

Implementar una pirámide Gaussiana (la imagen suavizada con cada $\sigma$ de una lista) y calcular las DoG entre niveles consecutivos.

- Ejercicio: implementar
    - construir_piramide_gaussiana
    - calcular_dogs

Ayudas:

- construir_piramide_gaussiana: una lista con `filters.gaussian(imagen, sigma=sigma)` por cada sigma.

- calcular_dogs: hay una DoG por cada par de gaussianas consecutivas, así que son una menos que gaussianas. Siempre es la más borrosa menos la menos borrosa.

In [ ]:
def construir_piramide_gaussiana(imagen, sigmas):
  # COMPLETAR: devolver una lista con la imagen suavizada con cada sigma
  raise NotImplementedError("Completar construir_piramide_gaussiana")

def calcular_dogs(gaussianas):
  # COMPLETAR: la gaussiana mas borrosa menos la menos borrosa, para cada par consecutivo
  raise NotImplementedError("Completar calcular_dogs")

### Tests (pirámide y DoG)

In [ ]:
def imagen_con_impulso(lado=41):
  imagen = np.zeros((lado, lado))
  imagen[lado // 2, lado // 2] = 1
  return imagen

# Un impulso suavizado es una gaussiana de pico 1 / (2 pi sigma^2), asi que el centro de la DoG es la diferencia de los picos
PICO_DOG_IMPULSO = 1 / (2 * np.pi * 2.0 ** 2) - 1 / (2 * np.pi * 1.0 ** 2)

def verificar_piramide_gaussiana():
  imagen = imagen_con_impulso()
  sigmas = [1.0, 2.0, 4.0]
  gaussianas = construir_piramide_gaussiana(imagen, sigmas)

  assert len(gaussianas) == len(sigmas), f"[piramide] tiene que haber una imagen por sigma ({len(sigmas)}) pero hay {len(gaussianas)}"
  assert all(gaussiana.shape == imagen.shape for gaussiana in gaussianas), "[piramide] cada imagen suavizada tiene que ser del mismo tamano que la original"
  assert np.isclose(gaussianas[0][20, 20], 1 / (2 * np.pi), atol=1e-3), f"[piramide] el centro con sigma = 1 tiene que valer ~{1 / (2 * np.pi):.4f}, se obtuvo {gaussianas[0][20, 20]:.4f}"
  assert gaussianas[0][20, 20] > gaussianas[1][20, 20] > gaussianas[2][20, 20], "[piramide] a mas sigma, mas borroso, asi que el pico del impulso tiene que bajar"
  print("OK [piramide] una imagen suavizada por cada sigma")

def verificar_dogs():
  gaussianas = construir_piramide_gaussiana(imagen_con_impulso(), [1.0, 2.0])
  dogs = calcular_dogs(gaussianas)
  assert len(dogs) == 1, f"[dogs] con 2 gaussianas tiene que haber 1 DoG, se obtuvo {len(dogs)}"
  assert np.isclose(dogs[0][20, 20], PICO_DOG_IMPULSO, atol=1e-3), f"[dogs] el centro de la DoG tiene que valer ~{PICO_DOG_IMPULSO:.4f} (mas borrosa menos nitida), se obtuvo {dogs[0][20, 20]:.4f}"
  assert np.isclose(dogs[0].sum(), 0, atol=1e-6), "[dogs] suavizar no cambia el brillo total, asi que cada DoG tiene que sumar ~0"

  dogs_constante = calcular_dogs(construir_piramide_gaussiana(np.ones((30, 30)), [1.0, 2.0, 4.0]))
  assert len(dogs_constante) == 2, f"[dogs] con 3 gaussianas tiene que haber 2 DoG, se obtuvo {len(dogs_constante)}"
  assert all(np.allclose(dog, 0, atol=1e-6) for dog in dogs_constante), "[dogs] una imagen constante no tiene estructuras: todas las DoG tienen que ser 0"
  print("OK [dogs] una DoG por cada par consecutivo, mas borrosa menos nitida")

### Verificacion piramide y DoG

In [ ]:
verificar_piramide_gaussiana()
verificar_dogs()

#### **Ejemplo**

Aplicamos en cameraman. Cada panel usa su propio rango, porque con $\sigma$ grande las diferencias son mucho más fuertes.

In [ ]:
#@title Función para graficar la pirámide y las DoG { display-mode: "form" }

def graficar_piramide_y_dogs(gaussianas, dogs, sigmas):
  fig, ejes = plt.subplots(2, len(gaussianas), figsize=(3.7 * len(gaussianas), 7.4))

  for columna, (gaussiana, sigma) in enumerate(zip(gaussianas, sigmas)):
    ejes[0, columna].imshow(gaussiana, cmap="gray", vmin=0, vmax=1)
    ejes[0, columna].set_title(f"$G_{{\\sigma = {sigma:g}}} * I$")
    ejes[0, columna].axis("off")

  for columna, (dog, sigma_nitida, sigma_borrosa) in enumerate(zip(dogs, sigmas[:-1], sigmas[1:])):
    limite = np.percentile(np.abs(dog), 99.7)
    ejes[1, columna].imshow(dog, cmap=MAPA_DIVERGENTE, vmin=-limite, vmax=limite)
    ejes[1, columna].set_title(f"DoG: $\\sigma = {sigma_borrosa:g}$ menos $\\sigma = {sigma_nitida:g}$")
    ejes[1, columna].axis("off")

  # El ultimo panel de abajo queda libre: lo usamos para explicar la escala de colores
  ejes[1, -1].axis("off")
  barra = fig.colorbar(plt.cm.ScalarMappable(cmap=MAPA_DIVERGENTE), cax=ejes[1, -1].inset_axes([0.05, 0.15, 0.07, 0.7]), ticks=[0, 0.5, 1])
  barra.ax.set_yticklabels(["Mas oscura que\nla nitida", "Igual", "Mas clara que\nla nitida"])

  fig.tight_layout()
  plt.show()

In [ ]:
camarografo = data.camera() / 255
sigmas = [1.0, 2.0, 4.0, 8.0]

gaussianas = construir_piramide_gaussiana(camarografo, sigmas)
dogs = calcular_dogs(gaussianas)

graficar_piramide_y_dogs(gaussianas, dogs, sigmas)

### **Cada DoG ve manchas de un tamaño**

Una mancha de radio $r$ responde más fuerte en la DoG cuyo $\sigma$ es parecido a $r/\sqrt{2}$. Para verlo armamos una imagen con tres discos de distinto tamaño y calculamos las DoG con muchos $\sigma$ consecutivos, cada uno un poco mayor que el anterior ($\sigma_{i+1} = 2^{1/4}\sigma_i$).

Movemos el slider para recorrer las escalas. A la derecha, cuánto responde la DoG en el centro de cada disco: **el disco grande se enciende recién con $\sigma$ grande**. Así es como SIFT sabe a qué escala está cada punto, y por eso un mismo punto se encuentra aunque la foto esté más cerca o más lejos.

In [ ]:
#@title Discos y respuesta por escala { display-mode: "form" }

RADIOS_DISCOS = (5, 10, 20)
CENTROS_DISCOS = [(40, 40), (40, 110), (110, 70)]  # (fila, columna)

def crear_imagen_de_discos(lado=160):
  filas, columnas = np.mgrid[:lado, :lado]
  imagen = np.zeros((lado, lado))
  for (fila_centro, columna_centro), radio in zip(CENTROS_DISCOS, RADIOS_DISCOS):
    imagen[(filas - fila_centro) ** 2 + (columnas - columna_centro) ** 2 <= radio ** 2] = 1
  return imagen

imagen_de_discos = crear_imagen_de_discos()
sigmas_de_discos = 2.0 ** (np.arange(0, 28) / 4)
dogs_de_discos = calcular_dogs(construir_piramide_gaussiana(imagen_de_discos, sigmas_de_discos))
respuestas_en_centros = [[abs(dog[fila, columna]) for dog in dogs_de_discos] for fila, columna in CENTROS_DISCOS]
limite_dogs_de_discos = max(np.abs(dog).max() for dog in dogs_de_discos)

def mostrar_discos(indice_sigma):
  fig, (ax_dog, ax_curvas) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})

  ax_dog.imshow(dogs_de_discos[indice_sigma], cmap=MAPA_DIVERGENTE, vmin=-limite_dogs_de_discos, vmax=limite_dogs_de_discos)
  ax_dog.set_title(f"DoG: $\\sigma = {sigmas_de_discos[indice_sigma + 1]:.1f}$ menos $\\sigma = {sigmas_de_discos[indice_sigma]:.1f}$")
  ax_dog.axis("off")

  sigmas_dibujados = sigmas_de_discos[1:]
  for radio, respuesta, color_disco in zip(RADIOS_DISCOS, respuestas_en_centros, COLORES_CLASES):
    ax_curvas.plot(sigmas_dibujados, respuesta, color=color_disco, linewidth=2, label=f"Disco de radio {radio}")
    ax_curvas.plot(sigmas_dibujados[indice_sigma], respuesta[indice_sigma], "o", color=color_disco, markeredgecolor="white", markersize=9, zorder=3)
    ax_curvas.axvline(radio / np.sqrt(2), color=color_disco, linestyle=":", linewidth=1.5)

  ax_curvas.set_xscale("log")
  ax_curvas.set_xlabel("$\\sigma$ (escala logaritmica)")
  ax_curvas.set_ylabel("|DoG| en el centro del disco")
  ax_curvas.set_title("Respuesta de cada disco (linea punteada: $r / \\sqrt{2}$)")
  ax_curvas.legend(frameon=False)

  fig.tight_layout()
  plt.show()

In [ ]:
interact(
  mostrar_discos,
  indice_sigma=IntSlider(min=0, max=len(dogs_de_discos) - 1, step=1, value=6, description="Escala", continuous_update=False),
);

## **Keypoints y descriptores**

`feature.SIFT` hace todo el proceso. Con `detect_and_extract` encontramos los keypoints y calculamos sus descriptores. Cada keypoint tiene:

> `positions`: dónde está (fila, columna).
\
> `sigmas`: su escala. Dibujamos un círculo de radio $\sqrt{2}\,\sigma$, el tamaño de la mancha que detectó.
\
> `orientations`: su orientación dominante, que dibujamos como una línea desde el centro.
\
> `descriptors`: los 128 números que lo describen.

In [ ]:
imagen_astronauta = color.rgb2gray(data.astronaut())

sift = feature.SIFT()
sift.detect_and_extract(imagen_astronauta)

print(f"Keypoints: {len(sift.positions)}")
print(f"Descriptores: {sift.descriptors.shape} (un vector de {sift.descriptors.shape[1]} numeros por keypoint)")

In [ ]:
#@title Funciones para dibujar keypoints y descriptores { display-mode: "form" }

def dibujar_keypoints(ax, posiciones, sigmas, orientaciones, color_keypoint=COLOR_CLASE_1, grosor=1.2):
  filas, columnas = posiciones[:, 0], posiciones[:, 1]
  radios = np.sqrt(2) * sigmas

  circulos = [Circle((columna, fila), radio) for fila, columna, radio in zip(filas, columnas, radios)]
  ax.add_collection(PatchCollection(circulos, facecolor="none", edgecolor=color_keypoint, linewidth=grosor))

  # skimage mide la orientacion desde el eje de las filas hacia el de las columnas
  extremos_x = columnas + radios * np.sin(orientaciones)
  extremos_y = filas + radios * np.cos(orientaciones)
  segmentos = [[(columna, fila), (extremo_x, extremo_y)] for columna, fila, extremo_x, extremo_y in zip(columnas, filas, extremos_x, extremos_y)]
  ax.add_collection(LineCollection(segmentos, colors=color_keypoint, linewidths=grosor))

def rotar_a_la_imagen(fila_ventana, columna_ventana, orientacion):
  # Pasa coordenadas de la ventana del descriptor (girada con el keypoint) a coordenadas de la imagen
  fila = np.cos(orientacion) * fila_ventana - np.sin(orientacion) * columna_ventana
  columna = np.sin(orientacion) * fila_ventana + np.cos(orientacion) * columna_ventana
  return fila, columna

def dibujar_descriptor(ax, imagen, posicion, sigma, orientacion, descriptor, celdas=4, orientaciones_por_celda=8):
  fila_centro, columna_centro = posicion
  histogramas = descriptor.reshape(celdas, celdas, orientaciones_por_celda).astype(float)

  # La ventana del descriptor mide 12 sigma de lado: 4 celdas de 3 sigma, giradas segun la orientacion del keypoint
  mitad_ventana = 6 * sigma
  esquinas = [(-mitad_ventana, -mitad_ventana), (-mitad_ventana, mitad_ventana), (mitad_ventana, mitad_ventana), (mitad_ventana, -mitad_ventana)]
  esquinas_en_imagen = [rotar_a_la_imagen(fila, columna, orientacion) for fila, columna in esquinas]
  poligono = [(columna_centro + columna, fila_centro + fila) for fila, columna in esquinas_en_imagen]

  margen = 9 * sigma
  ax.imshow(imagen, cmap="gray", vmin=0, vmax=1)
  ax.set_xlim(columna_centro - margen, columna_centro + margen)
  ax.set_ylim(fila_centro + margen, fila_centro - margen)
  ax.add_patch(Polygon(poligono, closed=True, fill=False, edgecolor=COLOR_CLASE_1, linewidth=2))

  # Cada celda dibuja su histograma como una rosa de 8 flechas (igual que las celdas de HOG)
  largo_maximo = 1.3 * sigma
  centros_celdas = (np.arange(1, celdas + 1) - (celdas + 1) / 2) * (2 * mitad_ventana / celdas)
  for indice_fila, fila_ventana in enumerate(centros_celdas):
    for indice_columna, columna_ventana in enumerate(centros_celdas):
      fila_celda, columna_celda = rotar_a_la_imagen(fila_ventana, columna_ventana, orientacion)
      x_celda, y_celda = columna_centro + columna_celda, fila_centro + fila_celda
      for indice_bin in range(orientaciones_por_celda):
        # El bin 0 es la direccion de la orientacion del keypoint; los demas avanzan de a 45 grados
        angulo = orientacion + 2 * np.pi * indice_bin / orientaciones_por_celda
        largo = largo_maximo * histogramas[indice_fila, indice_columna, indice_bin] / histogramas.max()
        ax.plot([x_celda, x_celda + largo * np.sin(angulo)], [y_celda, y_celda + largo * np.cos(angulo)], color=COLOR_CLASE_0, linewidth=1.6)

  # Esta flecha marca la orientacion del keypoint
  ax.plot([columna_centro, columna_centro + mitad_ventana * np.sin(orientacion)], [fila_centro, fila_centro + mitad_ventana * np.cos(orientacion)], color=COLOR_CLASE_1, linewidth=2)
  ax.axis("off")

Primero, los keypoints encontrados. Con el slider nos quedamos solo con los de $\sigma$ mayor a un mínimo: quedan las estructuras grandes (la cara, el casco) y desaparecen los detalles chicos.

In [ ]:
def mostrar_keypoints(sigma_minimo):
  seleccion = sift.sigmas >= sigma_minimo

  fig, ax = plt.subplots(figsize=(7.5, 7.5))
  ax.imshow(imagen_astronauta, cmap="gray")
  dibujar_keypoints(ax, sift.positions[seleccion], sift.sigmas[seleccion], sift.orientations[seleccion])
  ax.set_title(f"{seleccion.sum()} de {len(seleccion)} keypoints con $\\sigma \\geq$ {sigma_minimo:.1f}")
  ax.axis("off")
  plt.show()

interact(
  mostrar_keypoints,
  sigma_minimo=FloatSlider(min=0.5, max=20, step=0.5, value=3, description="$\\sigma$ minimo", continuous_update=False),
);

Ahora un keypoint de cerca. El cuadrado naranja es la ventana del descriptor, **girada según la orientación del keypoint** (la línea naranja): por eso el descriptor no cambia si rotamos la imagen. En cada una de sus 16 celdas, las 8 flechas azules son el histograma de orientaciones de gradiente, igual que en HOG.

In [ ]:
def mostrar_descriptor(indice_keypoint):
  fig, ax = plt.subplots(figsize=(6.5, 6.5))
  dibujar_descriptor(
    ax, imagen_astronauta,
    sift.positions[indice_keypoint], sift.sigmas[indice_keypoint], sift.orientations[indice_keypoint], sift.descriptors[indice_keypoint],
  )
  ax.set_title(f"Keypoint {indice_keypoint}: $\\sigma$ = {sift.sigmas[indice_keypoint]:.1f}")
  plt.show()

# Mostramos solo los keypoints grandes: con pocos pixeles las flechas no se distinguen
indices_grandes = np.flatnonzero(sift.sigmas >= 6)

interact(
  mostrar_descriptor,
  indice_keypoint=Dropdown(options=[int(indice) for indice in indices_grandes], value=int(indices_grandes[len(indices_grandes) // 2]), description="Keypoint"),
);

## **Matcheo de keypoints**

Si dos fotos muestran la misma escena, los keypoints de una deberían encontrarse en la otra con descriptores parecidos. Para probarlo creamos una segunda imagen **transformando la primera** (más chica, rotada y corrida), así sabemos exactamente dónde cae cada keypoint y podemos medir si los matcheos son correctos.

In [ ]:
imagen_1 = imagen_astronauta

# La transformacion que le aplicamos a la imagen: la vamos a querer recuperar mas abajo
transformacion_real = transform.SimilarityTransform(scale=0.8, rotation=np.deg2rad(25), translation=(50, -30))
imagen_2 = transform.warp(imagen_1, transformacion_real.inverse)

sift_1 = sift
sift_2 = feature.SIFT()
sift_2.detect_and_extract(imagen_2)

print(f"Keypoints en la imagen 1: {len(sift_1.positions)}, en la imagen 2: {len(sift_2.positions)}")

`match_descriptors` busca, para cada descriptor de la imagen 1, el más parecido de la imagen 2. Tiene dos filtros para quedarse solo con matcheos confiables:

> `cross_check`: el match tiene que ser el mejor **en los dos sentidos**. Si A elige a B pero B prefiere a C, se descarta.
> `max_ratio`: compara la distancia al vecino más cercano con la del segundo más cercano. Si son parecidas, el descriptor es ambiguo (podría ser cualquiera de los dos) y se descarta. Con valores chicos solo sobreviven los matches claros.

Como conocemos la transformación real, un match es **correcto** si el keypoint de la imagen 1, movido con esa transformación, cae a menos de 4 píxeles del keypoint que eligió en la imagen 2.

In [ ]:
#@title Funciones para evaluar y dibujar matches { display-mode: "form" }

TOLERANCIA_MATCH_CORRECTO = 4  # en pixeles

def en_xy(posiciones):
  # skimage guarda (fila, columna); para dibujar y transformar usamos (x, y)
  return posiciones[:, ::-1]

def matches_correctos(matches, transformacion):
  esperados = transformacion(en_xy(sift_1.positions[matches[:, 0]]))
  encontrados = en_xy(sift_2.positions[matches[:, 1]])
  return np.linalg.norm(esperados - encontrados, axis=1) < TOLERANCIA_MATCH_CORRECTO

def dibujar_matches(ax, matches, son_correctos, maximo_dibujados=120):
  ancho_imagen_1 = imagen_1.shape[1]
  ax.imshow(np.hstack([imagen_1, imagen_2]), cmap="gray")

  # Con cientos de lineas no se ve nada: dibujamos una muestra al azar, los conteos del titulo son de todos los matches
  elegidos = np.random.default_rng(SEMILLA).permutation(len(matches))[:maximo_dibujados]
  origenes = en_xy(sift_1.positions[matches[elegidos, 0]])
  destinos = en_xy(sift_2.positions[matches[elegidos, 1]]) + [ancho_imagen_1, 0]
  son_correctos = son_correctos[elegidos]

  # Dibujamos primero los incorrectos para que los correctos no queden tapados
  for correcto, color_linea in ((False, COLOR_CLASE_1), (True, COLOR_CLASE_0)):
    seleccion = son_correctos == correcto
    segmentos = [[origen, destino] for origen, destino in zip(origenes[seleccion], destinos[seleccion])]
    ax.add_collection(LineCollection(segmentos, colors=color_linea, linewidths=1.1, alpha=0.9))
  ax.axis("off")

def referencias_de_matches():
  return [
    Line2D([], [], color=COLOR_CLASE_0, linewidth=2, label="Match correcto"),
    Line2D([], [], color=COLOR_CLASE_1, linewidth=2, label="Match incorrecto"),
  ]

In [ ]:
def mostrar_matches(max_ratio, usar_cross_check):
  matches = feature.match_descriptors(sift_1.descriptors, sift_2.descriptors, cross_check=usar_cross_check, max_ratio=max_ratio)
  son_correctos = matches_correctos(matches, transformacion_real)

  fig, ax = plt.subplots(figsize=(13, 6.5))
  dibujar_matches(ax, matches, son_correctos)
  ax.set_title(f"{len(matches)} matches, {son_correctos.mean():.0%} correctos")
  ax.legend(handles=referencias_de_matches(), frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2)
  plt.show()

interact(
  mostrar_matches,
  max_ratio=FloatSlider(min=0.4, max=1.0, step=0.05, value=1.0, description="max_ratio", continuous_update=False),
  usar_cross_check=Checkbox(value=False, description="cross_check"),
);

Sin ningún filtro (`max_ratio = 1` y sin `cross_check`) cada keypoint de la imagen 1 se matchea con *algo*, y muchos matches están mal. Bajando `max_ratio` o activando `cross_check` se pierden matches pero casi todos los que quedan son correctos.

### De los matches a la transformación

Con matcheos confiables podemos hacer algo más: **estimar cómo se movió la imagen**. Los pocos matches incorrectos que quedan pueden arruinar el ajuste, así que usamos **RANSAC**: prueba muchas transformaciones armadas con pocos matches al azar y se queda con la que explica a más matches (los *inliers*).

In [ ]:
matches = feature.match_descriptors(sift_1.descriptors, sift_2.descriptors, cross_check=True, max_ratio=0.8)

transformacion_estimada, inliers = measure.ransac(
  (en_xy(sift_1.positions[matches[:, 0]]), en_xy(sift_2.positions[matches[:, 1]])),
  transform.SimilarityTransform, min_samples=3, residual_threshold=3, max_trials=1000,
)

def texto_traslacion(transformacion):
  traslacion_x, traslacion_y = transformacion.translation
  return f"({traslacion_x:.1f}, {traslacion_y:.1f})"

print(f"Inliers: {inliers.sum()} de {len(matches)} matches")
print(f"{'':12s}{'Real':>14s}{'Estimada':>14s}")
print(f"{'Escala':12s}{transformacion_real.scale:14.3f}{transformacion_estimada.scale:14.3f}")
print(f"{'Rotacion':12s}{np.rad2deg(transformacion_real.rotation):13.1f}°{np.rad2deg(transformacion_estimada.rotation):13.1f}°")
print(f"{'Traslacion':12s}{texto_traslacion(transformacion_real):>14s}{texto_traslacion(transformacion_estimada):>14s}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.5))
dibujar_matches(ax, matches, inliers)
ax.set_title(f"Matches que RANSAC acepta como inliers: {inliers.sum()} de {len(matches)}")
ax.legend(handles=[Line2D([], [], color=COLOR_CLASE_0, linewidth=2, label="Inlier"), Line2D([], [], color=COLOR_CLASE_1, linewidth=2, label="Outlier")], frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2)
plt.show()

# **Parte 2: Bag of Words**

El modelo **Bag of Words** nació en el procesamiento de texto: cada documento se representa como un vector con la **frecuencia de cada palabra**, sin importar el orden en que aparecen. Dos documentos que usan las mismas palabras se parecen, aunque estén escritos distinto.

Lo mismo se puede hacer con imágenes si definimos qué es una **palabra visual**. Los pasos son:

> 1. Extraemos los descriptores SIFT de todas las imágenes de entrenamiento.
> 2. Agrupamos esos descriptores con **k-means**. El centro de cada grupo es una palabra visual, y el conjunto de todas es el **corpus visual** (el "vocabulario").
> 3. Describimos cada imagen con un histograma: cuántos de sus descriptores caen en cada palabra.
> 4. Reducimos el peso de las palabras que aparecen en todas las imágenes (**tf-idf**).
> 5. Para buscar imágenes parecidas, comparamos sus vectores con la **similitud coseno**.

Imágenes parecidas van a tener vectores parecidos, así que las vamos a poder buscar y comparar con ejemplos que ya conocemos.

## **Imports**

In [ ]:
import os
import time

from joblib import Parallel, delayed
from tqdm.auto import tqdm
from scipy import ndimage
from datasets import load_dataset
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

import plotly.graph_objects as go
from plotly.subplots import make_subplots

## **El Dataset**

Usamos un subset de [**Imagenette**](https://huggingface.co/datasets/johnowhitaker/imagenette2-320), que a su vez es un subset de [ImageNet](https://www.image-net.org/): 10 clases de objetos fáciles de distinguir. Elegimos tres:

> **Iglesias**
> **Perros** (Springer spaniel inglés)
> **Caseteras** (reproductores de casetes)

Se descarga directo de Hugging Face (unos 350 MB la primera vez). Reducimos cada imagen a 128 píxeles de lado máximo y la pasamos a gris, que es lo único que usa SIFT.

Para poder **medir** qué tan bien funciona el método separamos las imágenes en dos grupos:

> **Train:** con estas armamos el vocabulario visual y es la "base" donde buscamos.
> **Validación:** el vocabulario nunca las vio. Son las imágenes que usamos como consulta.

In [ ]:
#@title Cargar el dataset { display-mode: "form" }

# Identificador de WordNet de cada clase de Imagenette y el nombre que le damos
CLASES_ELEGIDAS = {"n03028079": "iglesia", "n02102040": "perro", "n02979186": "casetera"}
NOMBRES_CLASES = list(CLASES_ELEGIDAS.values())

IMAGENES_POR_CLASE = 1000  #@param {type:"integer"}
PORCENTAJE_TRAIN = 0.75
LADO_IMAGEN = 128  #@param {type:"integer"}

def elegir_indices_por_clase(etiquetas, ids_clases, imagenes_por_clase, generador):
  indices, clases = [], []
  for clase, id_clase in enumerate(ids_clases):
    candidatos = np.flatnonzero(etiquetas == id_clase)
    indices.extend(generador.choice(candidatos, size=imagenes_por_clase, replace=False))
    clases.extend([clase] * imagenes_por_clase)
  return np.array(indices), np.array(clases)

def cargar_en_gris(imagen_pil, lado_maximo):
  gris = np.asarray(imagen_pil.convert("L"), dtype=float) / 255
  escala = lado_maximo / max(gris.shape)
  reducida = transform.resize(gris, (round(gris.shape[0] * escala), round(gris.shape[1] * escala)), anti_aliasing=True)

  # float32 ocupa la mitad de memoria, que importa al repartir miles de imagenes entre procesos
  return reducida.astype(np.float32)

def dividir_en_train_y_validacion(clases, porcentaje_train):
  es_train = np.zeros(len(clases), dtype=bool)
  for clase in np.unique(clases):
    indices_de_la_clase = np.flatnonzero(clases == clase)
    es_train[indices_de_la_clase[:round(len(indices_de_la_clase) * porcentaje_train)]] = True
  return es_train

dataset = load_dataset("johnowhitaker/imagenette2-320", split="train")
ids_clases = [dataset.features["label"].names.index(id_wordnet) for id_wordnet in CLASES_ELEGIDAS]

# Pedir solo la columna "label" evita decodificar las 13394 imagenes
indices, clases = elegir_indices_por_clase(np.array(dataset["label"]), ids_clases, IMAGENES_POR_CLASE, np.random.default_rng(SEMILLA))
imagenes = [cargar_en_gris(dataset[int(indice)]["image"], LADO_IMAGEN) for indice in tqdm(indices, desc="Cargando imagenes")]

es_train = dividir_en_train_y_validacion(clases, PORCENTAJE_TRAIN)
indices_train, indices_val = np.flatnonzero(es_train), np.flatnonzero(~es_train)
clases_train, clases_val = clases[indices_train], clases[indices_val]

print(f"Imagenes: {len(imagenes)} ({len(indices_train)} de train y {len(indices_val)} de validacion)")

In [ ]:
#@title Funcion para graficar ejemplos { display-mode: "form" }

def graficar_ejemplos(imagenes, clases, por_clase=7):
  fig, ejes = plt.subplots(len(NOMBRES_CLASES), por_clase, figsize=(1.9 * por_clase, 2.0 * len(NOMBRES_CLASES)))
  for clase, nombre in enumerate(NOMBRES_CLASES):
    for columna, indice in enumerate(np.flatnonzero(clases == clase)[:por_clase]):
      ejes[clase, columna].imshow(imagenes[indice], cmap="gray", vmin=0, vmax=1)
      ejes[clase, columna].axis("off")
    ejes[clase, 0].text(-0.08, 0.5, nombre, transform=ejes[clase, 0].transAxes, ha="right", va="center", fontweight="bold", color=COLORES_CLASES[clase], fontsize=13)
  fig.subplots_adjust(left=0.1, wspace=0.05, hspace=0.05)
  plt.show()

In [ ]:
graficar_ejemplos(imagenes, clases)

## **Recolección de Palabras Visuales o creación del Corpus Visual**

### **Recolección de Features**

Por cada una de las imágenes calculamos sus keypoints y descriptores con **SIFT**, como en la parte 1. Es lo más lento de todo el laboratorio y se hace una sola vez: usamos `joblib` para repartir las imágenes entre los procesadores que tenga la máquina.

Cada imagen aporta una cantidad distinta de keypoints, según cuánta textura tenga.

In [ ]:
def extraer_sift(imagen):
  sift = feature.SIFT()

  # SIFT levanta un error si una imagen no tiene ningun keypoint (una imagen casi lisa, por ejemplo)
  try:
    sift.detect_and_extract(imagen)
  except RuntimeError:
    return np.zeros((0, 2)), np.zeros(0), np.zeros(0), np.zeros((0, 128), dtype=np.uint8)
  return sift.positions, sift.sigmas, sift.orientations, sift.descriptors

inicio = time.time()
resultados = list(tqdm(
  Parallel(n_jobs=-1, return_as="generator")(delayed(extraer_sift)(imagen) for imagen in imagenes),
  total=len(imagenes), desc="Extrayendo features",
))

# Una lista por cada dato que guarda SIFT, con un elemento por imagen
posiciones, sigmas, orientaciones, descriptores = (list(dato_por_imagen) for dato_por_imagen in zip(*resultados))

print(f"{time.time() - inicio:.0f} s, {np.mean([len(descriptores_de_imagen) for descriptores_de_imagen in descriptores]):.0f} keypoints por imagen en promedio")

In [ ]:
fig, ejes = plt.subplots(1, len(NOMBRES_CLASES), figsize=(13, 4.4))
for clase, ax in enumerate(ejes):
  indice = indices_train[np.flatnonzero(clases_train == clase)[3]]
  ax.imshow(imagenes[indice], cmap="gray", vmin=0, vmax=1)
  dibujar_keypoints(ax, posiciones[indice], sigmas[indice], orientaciones[indice], color_keypoint=COLOR_CLASE_1, grosor=1)
  ax.set_title(f"{NOMBRES_CLASES[clase]}: {len(posiciones[indice])} keypoints", color=COLORES_CLASES[clase])
  ax.axis("off")
fig.tight_layout()
plt.show()

Para armar el vocabulario nos quedamos **solo con los descriptores de las imágenes de train**, juntados en una gran matriz de una fila por descriptor. Las imágenes de validación no pueden influir en el vocabulario: si lo hicieran, las consultas ya no serían "imágenes nuevas".

In [ ]:
matriz_descriptores_train = np.concatenate([descriptores[indice] for indice in indices_train]).astype(np.float32)

matriz_descriptores_train.shape

### **Calculamos nuestras palabras visuales**

Nuestras palabras visuales serán el **centroide** de los clusters de características encontrados en el dataset de train.

Usamos `MiniBatchKMeans` de scikit-learn, una versión de k-means que actualiza los centroides con tandas chicas de descriptores. Tarda segundos y no horas, y con tantos descriptores el resultado es casi el mismo. El parámetro `K` es en cuántos clusters dividimos, es decir, **el tamaño de nuestro vocabulario**.

In [ ]:
K = 200  #@param {type:"integer"}

inicio = time.time()
modelo_kmeans = MiniBatchKMeans(n_clusters=K, n_init=3, batch_size=4096, random_state=SEMILLA).fit(matriz_descriptores_train)

# Cada fila es una palabra visual: un descriptor "promedio" de 128 numeros
corpus = modelo_kmeans.cluster_centers_

print(f"{time.time() - inicio:.0f} s, corpus de {corpus.shape[0]} palabras de {corpus.shape[1]} numeros")

### **¿Cómo se ve una palabra visual?**

Un centroide son 128 números y no se puede mirar. Pero sí podemos mirar los **parches de las imágenes** cuyos descriptores cayeron en esa palabra. Para cada palabra mostramos los 12 más cercanos al centroide. Los recortamos igual que SIFT: una ventana de $12\sigma$ de lado, **girada según la orientación del keypoint**, que es justo lo que "ve" el descriptor.

Cada fila es una palabra: los parches de una fila se parecen entre sí (un borde, una esquina, una textura de líneas, una mancha) aunque vengan de imágenes distintas. Con el slider elegimos cualquier palabra del vocabulario.

In [ ]:
#@title Funciones para mostrar las palabras visuales { display-mode: "form" }

# Para cada descriptor de train: de que imagen vino, donde estaba y en que palabra cae
imagen_de_cada_descriptor = np.concatenate([np.full(len(descriptores[indice]), indice) for indice in indices_train])
posicion_de_cada_descriptor = np.concatenate([posiciones[indice] for indice in indices_train])
sigma_de_cada_descriptor = np.concatenate([sigmas[indice] for indice in indices_train])
orientacion_de_cada_descriptor = np.concatenate([orientaciones[indice] for indice in indices_train])
palabra_de_cada_descriptor = modelo_kmeans.predict(matriz_descriptores_train)

def extraer_parche_de_ventana(imagen, posicion, sigma, orientacion, lado=24):
  # Muestreamos la ventana del descriptor (12 sigma de lado, girada con el keypoint) sobre una grilla de lado x lado pixeles
  coordenadas = (np.arange(lado) - (lado - 1) / 2) * (12 * sigma / lado)
  fila_ventana, columna_ventana = np.meshgrid(coordenadas, coordenadas, indexing="ij")
  fila, columna = rotar_a_la_imagen(fila_ventana, columna_ventana, orientacion)
  parche = ndimage.map_coordinates(imagen, [posicion[0] + fila, posicion[1] + columna], order=1, mode="nearest")

  # SIFT no depende del brillo ni del contraste, asi que estiramos cada parche para verlo bien
  return (parche - parche.min()) / (parche.max() - parche.min() + 1e-6)

def graficar_palabras_visuales(palabras, parches_por_palabra=12):
  fig, ejes = plt.subplots(len(palabras), parches_por_palabra, figsize=(0.95 * parches_por_palabra + 1.8, 1.0 * len(palabras) + 0.2), squeeze=False)
  for fila, palabra in enumerate(palabras):
    descriptores_de_la_palabra = np.flatnonzero(palabra_de_cada_descriptor == palabra)
    distancias_al_centroide = np.linalg.norm(matriz_descriptores_train[descriptores_de_la_palabra] - corpus[palabra], axis=1)
    mas_cercanos = descriptores_de_la_palabra[np.argsort(distancias_al_centroide)[:parches_por_palabra]]

    for columna, ax in enumerate(ejes[fila]):
      ax.set_xticks([])
      ax.set_yticks([])
      for borde in ax.spines.values(): borde.set_visible(False)
      if columna >= len(mas_cercanos): continue

      descriptor = mas_cercanos[columna]
      parche = extraer_parche_de_ventana(
        imagenes[imagen_de_cada_descriptor[descriptor]], posicion_de_cada_descriptor[descriptor],
        sigma_de_cada_descriptor[descriptor], orientacion_de_cada_descriptor[descriptor],
      )
      ax.imshow(parche, cmap="gray", vmin=0, vmax=1)

    ejes[fila, 0].set_ylabel(f"Palabra {palabra}\n{len(descriptores_de_la_palabra)} descriptores", rotation=0, ha="right", va="center", fontsize=10)
  fig.tight_layout()
  plt.show()

In [ ]:
# Ocho palabras repartidas entre las mas y las menos usadas del vocabulario
palabras_ordenadas_por_uso = np.argsort(-np.bincount(palabra_de_cada_descriptor, minlength=K))
graficar_palabras_visuales(palabras_ordenadas_por_uso[::K // 8][:8])

In [ ]:
interact(
  lambda palabra: graficar_palabras_visuales([palabra]),
  palabra=IntSlider(min=0, max=K - 1, step=1, value=0, description="Palabra", continuous_update=False),
);

## **Descripción de nuestras imágenes**

Ahora que tenemos nuestro **corpus visual**, nos queda utilizarlo para describir las imágenes, **tanto las de train como las de validación**.

Para esto clasificamos los descriptores de cada imagen según el corpus, y describimos a la imagen con la **frecuencia** en que aparece cada palabra visual.

### **Clasificación de los Descriptores**

Para clasificar un descriptor elegimos el centroide que más cerca está. Es lo que hace `predict`.

In [ ]:
# Clasificamos los descriptores de todas las imagenes juntos y despues los repartimos otra vez por imagen
descriptores_de_todas = np.concatenate(descriptores).astype(np.float32)
palabras_de_todas = modelo_kmeans.predict(descriptores_de_todas)
palabras_por_imagen = np.split(palabras_de_todas, np.cumsum([len(descriptores_de_imagen) for descriptores_de_imagen in descriptores])[:-1])

# Algunas palabras visuales de la imagen 0: lo que vemos es el indice de la palabra en el corpus
palabras_por_imagen[0][:5], len(palabras_por_imagen[0])

### **Frecuencia de las palabras**

Ahora que clasificamos los descriptores de cada imagen, la describimos como la **frecuencia de cada una de las palabras**: un vector de largo `K`. La matriz tiene una fila por imagen y una columna por palabra.

In [ ]:
frecuencias = np.stack([np.bincount(palabras_de_imagen, minlength=K) for palabras_de_imagen in palabras_por_imagen])
frecuencias_train, frecuencias_val = frecuencias[indices_train], frecuencias[indices_val]

frecuencias.shape

In [ ]:
# Comparamos con los valores que habiamos encontrado antes para la imagen 0.
for palabra in palabras_por_imagen[0][:5]:
  print(f"{palabra}: {frecuencias[0][palabra]}")

In [ ]:
#@title Funciones para graficar las frecuencias { display-mode: "form" }

def graficar_matriz_de_frecuencias(matriz, clases, titulo):
  fig, (ax_clases, ax_matriz) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 40], "wspace": 0.01})

  ax_matriz.imshow(matriz, aspect="auto", interpolation="nearest", cmap=MAPA_SECUENCIAL, vmax=np.percentile(matriz, 99))
  ax_matriz.set_title(titulo)
  ax_matriz.set_xlabel("Palabra visual")
  ax_matriz.set_yticks([])

  # La banda de la izquierda dice de que clase es cada fila (las filas estan ordenadas por clase)
  ax_clases.imshow(clases[:, None], aspect="auto", interpolation="nearest", cmap=LinearSegmentedColormap.from_list("clases", COLORES_CLASES, N=len(COLORES_CLASES)))
  ax_clases.set_xticks([])
  ax_clases.set_yticks([np.flatnonzero(clases == clase).mean() for clase in range(len(NOMBRES_CLASES))])
  ax_clases.set_yticklabels(NOMBRES_CLASES)
  for etiqueta, color_clase in zip(ax_clases.get_yticklabels(), COLORES_CLASES): etiqueta.set_color(color_clase)
  for borde in ax_clases.spines.values(): borde.set_visible(False)
  plt.show()

def graficar_histogramas_de_ejemplo(matriz, clases, titulo):
  fig, ejes = plt.subplots(1, len(NOMBRES_CLASES), figsize=(14, 3.2), sharey=True)
  for clase, ax in enumerate(ejes):
    indice = np.flatnonzero(clases == clase)[3]
    ax.bar(np.arange(matriz.shape[1]), matriz[indice], color=COLORES_CLASES[clase], width=1.0)
    ax.set_title(f"{NOMBRES_CLASES[clase]}", color=COLORES_CLASES[clase])
    ax.set_xlabel("Palabra visual")
  ejes[0].set_ylabel(titulo)
  fig.tight_layout()
  plt.show()

In [ ]:
graficar_histogramas_de_ejemplo(frecuencias_train, clases_train, "Frecuencia")
graficar_matriz_de_frecuencias(frecuencias_train, clases_train, "Frecuencias: una fila por imagen de train")

Cada imagen es un histograma: un vector de `K` números. En la matriz de abajo, cada fila es una imagen (ordenadas por clase) y cada columna una palabra visual. Si hay columnas más oscuras para una clase que para las otras, esa palabra ayuda a distinguirla.

## **Tf-idf**

Como en **texto**, algunas palabras serán más importantes que otras para describir el contenido de una imagen. Por ejemplo, la palabra "***entonces***" no es útil por sí sola para describir un escrito, ya que varios documentos de contenidos completamente opuestos pueden contener esta palabra.

Por estas razones, reduciremos la importancia de las palabras que aparecen en la mayoría de nuestras imágenes, ya que una palabra visual que toda imagen contiene no sirve para diferenciarlas. **Para esto utilizaremos la fórmula Tf-idf**

$$
\text{Tf-idf}_{t,d} = \text{tf}_{t,d} \cdot \log\left(\frac{N}{\text{df}_t}\right)
$$
donde
- $\text{Tf-idf}_{t,d}$ será el nuevo valor de la palabra $t$ para la imagen $d$
- $\text{tf}_{t,d}$ es la frecuencia de la palabra $t$ para la imagen $d$
- $N$ es la cantidad de imágenes totales
- $\text{df}_t$ es la cantidad de imágenes en la que la palabra $t$ aparece al menos una vez.

El factor $\log(N / \text{df}_t)$ es el **idf** de la palabra: vale 0 si aparece en todas las imágenes y crece cuanto más rara es.

Un detalle importante: $N$ y $\text{df}_t$ se calculan **solo con las imágenes de train**. Las imágenes de validación se multiplican por el mismo idf, porque tienen que tratarse como imágenes nuevas.

### ***Ejercicio***: Aplicar Tf-idf a la matriz de frecuencias

- Ejercicio: implementar
    - calcular_idf
    - aplicar_tfidf

Ayudas:

- calcular_idf: recibe la matriz de frecuencias (una fila por imagen) y devuelve un vector de largo `K` con el idf de cada palabra. $\text{df}_t$ sale de contar, por columna, en cuántas filas hay un valor mayor a 0. Si una palabra no aparece en ninguna imagen, $\text{df}_t$ sería 0 y el cociente no se puede calcular: como su frecuencia es 0 en todas las imágenes, da lo mismo qué idf le pongamos, así que usen $\text{df}_t = 1$ para evitar la división por cero.

- aplicar_tfidf: multiplica cada columna de la matriz por el idf de esa palabra. Mantengan el mismo orden de imágenes que en la matriz de frecuencias.

In [ ]:
def calcular_idf(frecuencias):
  # COMPLETAR: log(N / df) para cada palabra, con df = en cuantas imagenes aparece (minimo 1)
  raise NotImplementedError("Completar calcular_idf")

def aplicar_tfidf(frecuencias, idf):
  # COMPLETAR: tf * idf, manteniendo el orden de las imagenes
  raise NotImplementedError("Completar aplicar_tfidf")

### Tests (tf-idf)

In [ ]:
# nombre: (frecuencias, idf esperado)
IDF_ESPERADOS = {
  # df = [2, 1, 3, 0]: la ultima palabra no aparece nunca y se trata como df = 1
  'cuatro_imagenes':    (np.array([[3, 0, 1, 0], [0, 0, 2, 0], [1, 5, 0, 0], [0, 0, 4, 0]]), np.log([2, 4, 4 / 3, 4])),
  'palabra_en_todas':   (np.array([[1, 0], [2, 5], [3, 0]]),                                  np.log([1, 3])),  # la que esta en todas pesa 0
}

# nombre: (frecuencias, idf, tf-idf esperado)
TFIDF_ESPERADOS = {
  'se_multiplica_cada_columna': (np.array([[3, 0, 1], [0, 2, 4]]), np.array([0.5, 1.0, 2.0]), np.array([[1.5, 0.0, 2.0], [0.0, 2.0, 8.0]])),
  'idf_cero_anula_la_palabra':  (np.array([[7, 1], [2, 1]]),       np.array([0.0, 3.0]),      np.array([[0.0, 3.0], [0.0, 3.0]])),
}

def verificar_idf(nombre):
  frecuencias_de_prueba, esperado = IDF_ESPERADOS[nombre]
  obtenido = calcular_idf(frecuencias_de_prueba)

  assert np.shape(obtenido) == esperado.shape, f"[{nombre}] el idf tiene que tener un valor por palabra {esperado.shape}, se obtuvo {np.shape(obtenido)}"
  assert np.all(np.isfinite(obtenido)), f"[{nombre}] el idf tiene que ser finito, se obtuvo {obtenido}. Ojo con las palabras que no aparecen en ninguna imagen"
  assert np.allclose(obtenido, esperado), f"[{nombre}] se esperaba {np.round(esperado, 3)} pero se obtuvo {np.round(obtenido, 3)}"
  print(f"OK [{nombre}] idf = {np.round(obtenido, 3)}")

def verificar_tfidf(nombre):
  frecuencias_de_prueba, idf, esperado = TFIDF_ESPERADOS[nombre]
  obtenido = aplicar_tfidf(frecuencias_de_prueba, idf)

  assert np.shape(obtenido) == esperado.shape, f"[{nombre}] el resultado tiene que tener la misma forma que las frecuencias {esperado.shape}, se obtuvo {np.shape(obtenido)}"
  assert np.allclose(obtenido, esperado), f"[{nombre}] se esperaba\n{esperado}\npero se obtuvo\n{obtenido}"
  print(f"OK [{nombre}]")

### Verificacion idf

In [ ]:
for nombre in IDF_ESPERADOS:
  verificar_idf(nombre)

### Verificacion tf-idf

In [ ]:
for nombre in TFIDF_ESPERADOS:
  verificar_tfidf(nombre)

In [ ]:
# tfidf sera la nueva matriz de frecuencias habiendo aplicado tf-idf, en el mismo orden de imagenes que en frecuencias.
# El idf sale solo de train y se usa tal cual en validacion
idf = calcular_idf(frecuencias_train)
tfidf_train = aplicar_tfidf(frecuencias_train, idf)
tfidf_val = aplicar_tfidf(frecuencias_val, idf)

tfidf_train.shape, tfidf_train[0][:5]

In [ ]:
#@title Funcion para graficar el efecto del idf { display-mode: "form" }

def graficar_efecto_del_idf(frecuencias_train, idf, indice_ejemplo):
  imagenes_con_la_palabra = (frecuencias_train > 0).mean(axis=0)
  ejemplo_frecuencias = frecuencias_train[indice_ejemplo]
  ejemplo_tfidf = ejemplo_frecuencias * idf
  clase = clases_train[indice_ejemplo]

  fig, (ax_idf, ax_frecuencias, ax_tfidf) = plt.subplots(1, 3, figsize=(15, 3.6))

  ax_idf.scatter(imagenes_con_la_palabra, idf, s=22, color=COLOR_TINTA, alpha=0.6, edgecolor="white", linewidth=0.4)
  ax_idf.set_title("Peso de cada palabra (idf)")
  ax_idf.set_xlabel("Fraccion de imagenes que la contienen")
  ax_idf.set_ylabel("idf")

  for ax, valores, titulo in ((ax_frecuencias, ejemplo_frecuencias, "Frecuencias"), (ax_tfidf, ejemplo_tfidf, "Tf-idf")):
    ax.bar(np.arange(len(valores)), valores, color=COLORES_CLASES[clase], width=1.0)
    ax.set_title(f"{titulo} de una imagen de {NOMBRES_CLASES[clase]}")
    ax.set_xlabel("Palabra visual")

  fig.tight_layout()
  plt.show()

In [ ]:
graficar_efecto_del_idf(frecuencias_train, idf, indice_ejemplo=3)

En el primer gráfico, cuanto más imágenes contienen una palabra (más a la derecha), menos pesa. Comparando los dos histogramas, las palabras comunes se achican respecto de las raras y dejan de dominar el vector.

# **Búsqueda de imágenes según su descriptor**

Ahora que cada imagen está descrita utilizando Bag Of Words, podemos tomar una medida de su semejanza calculando la **similitud coseno** entre sus descriptores.

$$ \text{similitud coseno} = \frac{A \cdot B}{|A| \cdot |B|}$$

Esta medida nos habla sobre el ángulo entre los dos vectores de características: cuanto más cercano a 1, más similar será el contenido de las imágenes que describen. Como solo mira el ángulo, no le importa cuántos keypoints tenga cada imagen.

Buscamos con cada imagen de **validación** como consulta, en la base de imágenes de **train**.

In [ ]:
EPSILON = 1e-12  # para no dividir por cero si una imagen quedo sin keypoints

# Dividir cada fila por su norma deja vectores de largo 1: el coseno entre dos vectores es su producto escalar
def normalizar_filas(matriz):
  return matriz / (np.linalg.norm(matriz, axis=1, keepdims=True) + EPSILON)

def similitud_coseno(consultas, base):
  return normalizar_filas(consultas) @ normalizar_filas(base).T

def buscar_vecinos(consultas, base, top_k):
  similitudes = similitud_coseno(consultas, base)

  # Elegimos los k indices con mejor puntaje, de mayor a menor similitud
  indices_vecinos = np.argsort(-similitudes, axis=1)[:, :top_k]
  return indices_vecinos, np.take_along_axis(similitudes, indices_vecinos, axis=1)

In [ ]:
#@title Funcion para graficar una busqueda { display-mode: "form" }

def graficar_busqueda(indice_consulta, consultas, base, top_k=5, titulo=None):
  indices_vecinos, similitudes = buscar_vecinos(consultas[indice_consulta][None, :], base, top_k)
  clase_consulta = clases_val[indice_consulta]

  fig, ejes = plt.subplots(1, top_k + 1, figsize=(2.6 * (top_k + 1), 3.3))
  ejes[0].imshow(imagenes[indices_val[indice_consulta]], cmap="gray", vmin=0, vmax=1)
  ejes[0].set_title(f"Consulta: {NOMBRES_CLASES[clase_consulta]}", color=COLORES_CLASES[clase_consulta], fontsize=11)
  color_marco = COLORES_CLASES[clase_consulta]

  for ax, indice_vecino, similitud in zip(ejes[1:], indices_vecinos[0], similitudes[0]):
    clase_vecino = clases_train[indice_vecino]
    ax.imshow(imagenes[indices_train[indice_vecino]], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{NOMBRES_CLASES[clase_vecino]} {similitud:.2f} {'✓' if clase_vecino == clase_consulta else '✗'}", color=COLORES_CLASES[clase_vecino], fontsize=11)
    color_marco = COLORES_CLASES[clase_vecino]
    for borde in ax.spines.values():
      borde.set_visible(True)
      borde.set_edgecolor(color_marco)
      borde.set_linewidth(3)

  for borde in ejes[0].spines.values():
    borde.set_visible(True)
    borde.set_edgecolor(COLORES_CLASES[clase_consulta])
    borde.set_linewidth(3)

  for ax in ejes:
    ax.set_xticks([])
    ax.set_yticks([])
  if titulo is not None: fig.suptitle(titulo, x=0.01, ha="left", fontweight="bold")
  fig.tight_layout()
  plt.show()

Elegimos una imagen de validación y vemos sus 5 vecinos más parecidos de la base de train. El color y el nombre de cada imagen indican su **clase real**, y ✓ o ✗ si coincide con la de la consulta. Los números son la similitud coseno.

Probá con distintas imágenes, y cambiá entre **frecuencias** y **tf-idf**.

In [ ]:
representaciones = {
  "Frecuencias": (frecuencias_val.astype(float), frecuencias_train.astype(float)),
  "Tf-idf": (tfidf_val, tfidf_train),
}

def mostrar_busqueda(indice_consulta, representacion):
  consultas, base = representaciones[representacion]
  graficar_busqueda(indice_consulta, consultas, base, titulo=representacion)

interact(
  mostrar_busqueda,
  indice_consulta=IntSlider(min=0, max=len(indices_val) - 1, step=1, value=0, description="Consulta", continuous_update=False),
  representacion=Dropdown(options=list(representaciones), value="Tf-idf", description="Vectores"),
);

### **¿Qué tan bien funciona?**

Mirar ejemplos de a uno no alcanza. Para medirlo usamos **todas** las imágenes de validación como consulta y calculamos la **precisión en k** (*precision@k*): de los $k$ vecinos más parecidos, qué fracción es de la misma clase que la consulta.

Con 3 clases balanceadas, un buscador que elige al azar acierta un tercio de las veces: ese es el piso contra el que hay que comparar.

In [ ]:
def precision_en_k(consultas, clases_consultas, base, clases_base, top_k=5):
  indices_vecinos, _ = buscar_vecinos(consultas, base, top_k)
  return (clases_base[indices_vecinos] == clases_consultas[:, None]).mean()

def matriz_de_confusion_del_vecino_mas_cercano(consultas, clases_consultas, base, clases_base):
  indices_vecinos, _ = buscar_vecinos(consultas, base, 1)
  matriz = np.zeros((len(NOMBRES_CLASES), len(NOMBRES_CLASES)), dtype=int)
  for clase_consulta, clase_vecino in zip(clases_consultas, clases_base[indices_vecinos[:, 0]]): matriz[clase_consulta, clase_vecino] += 1
  return matriz

In [ ]:
#@title Funcion para graficar la evaluacion { display-mode: "form" }

def tonos_de_representaciones(cantidad):
  # Un tono de gris por cada representacion, de claro a oscuro: asi no se confunden con los colores de las clases
  escala_de_grises = LinearSegmentedColormap.from_list("tonos", ["#C3CBD4", COLOR_TINTA])
  return [escala_de_grises(posicion) for posicion in np.linspace(0, 1, max(cantidad, 2))[:cantidad]]

def graficar_evaluacion(representaciones, top_k=5):
  clases_nombres = NOMBRES_CLASES + ["todas"]
  ancho_barra = 0.8 / len(representaciones)
  tonos = tonos_de_representaciones(len(representaciones))

  fig, (ax_barras, ax_confusion) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})

  for numero, (nombre, (consultas, base)) in enumerate(representaciones.items()):
    precisiones = [precision_en_k(consultas[clases_val == clase], clases_val[clases_val == clase], base, clases_train, top_k) for clase in range(len(NOMBRES_CLASES))]
    precisiones.append(precision_en_k(consultas, clases_val, base, clases_train, top_k))

    posiciones_barras = np.arange(len(clases_nombres)) + (numero - (len(representaciones) - 1) / 2) * ancho_barra
    ax_barras.bar(posiciones_barras, precisiones, width=ancho_barra * 0.92, color=tonos[numero], label=nombre)
    for posicion, precision in zip(posiciones_barras, precisiones): ax_barras.text(posicion, precision + 0.01, f"{precision:.2f}", ha="center", fontsize=8 if len(representaciones) > 3 else 9)

  ax_barras.axhline(1 / len(NOMBRES_CLASES), color=COLOR_TINTA, linestyle="--", linewidth=1, label="Al azar")
  ax_barras.set_xticks(np.arange(len(clases_nombres)))
  ax_barras.set_xticklabels(clases_nombres)
  ax_barras.set_ylim(0, 1.12)
  ax_barras.set_title(f"Precision@{top_k} segun la clase de la consulta")
  ax_barras.legend(frameon=False, loc="upper left", ncol=min(len(representaciones) + 1, 3))

  # La confusion se muestra para la ultima representacion: es la que queremos mirar mas de cerca
  nombre_final, (consultas_finales, base_final) = list(representaciones.items())[-1]
  confusion = matriz_de_confusion_del_vecino_mas_cercano(consultas_finales, clases_val, base_final, clases_train)
  ax_confusion.imshow(confusion, cmap=MAPA_SECUENCIAL)
  for clase_real in range(len(NOMBRES_CLASES)):
    for clase_vecino in range(len(NOMBRES_CLASES)):
      ax_confusion.text(clase_vecino, clase_real, confusion[clase_real, clase_vecino], ha="center", va="center", color="white" if confusion[clase_real, clase_vecino] > confusion.max() / 2 else COLOR_TINTA)
  ax_confusion.set_xticks(range(len(NOMBRES_CLASES)))
  ax_confusion.set_xticklabels(NOMBRES_CLASES)
  ax_confusion.set_yticks(range(len(NOMBRES_CLASES)))
  ax_confusion.set_yticklabels(NOMBRES_CLASES)
  ax_confusion.set_xlabel("Clase del vecino mas cercano")
  ax_confusion.set_ylabel("Clase de la consulta")
  ax_confusion.set_title(f"Vecino mas cercano ({nombre_final})")

  fig.tight_layout()
  plt.show()

In [ ]:
graficar_evaluacion(representaciones)

Lo que se ve:

> Bag of Words **supera claramente al azar**, pero está lejos de ser perfecto: cerca de 4 de cada 10 vecinos son de otra clase.
\
> Hay clases más fáciles que otras. La matriz de confusión muestra con qué clase se confunde cada una.
\
> **Tf-idf y frecuencias dan resultados muy parecidos.** El idf achica las palabras muy comunes, pero con este vocabulario ninguna palabra aparece en la gran mayoría de las imágenes (se ve en el primer gráfico del idf), así que el cambio es chico. Tf-idf pesa más cuando el vocabulario tiene palabras que aparecen en casi todo.

Medir con todas las consultas nos evita sacar conclusiones de un ejemplo que salió bien o mal.

## **Visualizando el espacio de características**

Cada imagen es un punto en un espacio de `K` dimensiones, imposible de dibujar. Con **PCA** buscamos las direcciones en las que los puntos más se dispersan y los proyectamos sobre las 2 o 3 principales: es la mejor "foto" plana de ese espacio.

Si los vecinos más cercanos fueran casi siempre de la misma clase, veríamos cada clase agrupada. El PCA se ajusta **solo con train**; las imágenes de validación se proyectan sobre las mismas direcciones y las dibujamos más grandes y con borde blanco.

Antes de proyectar **normalizamos cada vector** (largo 1), que es lo que hace por dentro la similitud coseno.

In [ ]:
#@title Funciones para graficar el espacio con PCA { display-mode: "form" }

def proyectar_con_pca(vectores_train, vectores_val, n_componentes=3):
  pca = PCA(n_components=n_componentes, random_state=SEMILLA).fit(vectores_train)
  return pca.transform(vectores_train), pca.transform(vectores_val), pca.explained_variance_ratio_

def graficar_pca_2d(espacios):
  fig, ejes = plt.subplots(1, len(espacios), figsize=(6.6 * len(espacios), 5.6), squeeze=False)
  for ax, (nombre, (vectores_train, vectores_val)) in zip(ejes[0], espacios.items()):
    proyeccion_train, proyeccion_val, varianza = proyectar_con_pca(vectores_train, vectores_val, 2)
    for clase in range(len(NOMBRES_CLASES)):
      ax.scatter(*proyeccion_train[clases_train == clase].T, s=16, alpha=0.35, color=COLORES_CLASES[clase], marker=MARCADORES_CLASES[clase], linewidth=0)
      ax.scatter(*proyeccion_val[clases_val == clase].T, s=48, alpha=0.95, color=COLORES_CLASES[clase], marker=MARCADORES_CLASES[clase], edgecolor="white", linewidth=0.8, zorder=3)
    ax.set_title(nombre)
    ax.set_xlabel(f"PC1 ({varianza[0]:.0%} de la varianza)")
    ax.set_ylabel(f"PC2 ({varianza[1]:.0%})")
    ax.set_aspect("equal", adjustable="datalim")

  referencias = [Line2D([], [], marker=MARCADORES_CLASES[clase], linestyle="", color=COLORES_CLASES[clase], markeredgecolor="white", markersize=9, label=nombre) for clase, nombre in enumerate(NOMBRES_CLASES)]
  referencias.append(Line2D([], [], marker="o", linestyle="", color=COLOR_TINTA, alpha=0.35, markersize=5, label="train (chico, claro)"))
  referencias.append(Line2D([], [], marker="o", linestyle="", color=COLOR_TINTA, markeredgecolor="white", markersize=9, label="validacion (grande)"))
  fig.legend(handles=referencias, loc="lower center", ncol=len(referencias), frameon=False, bbox_to_anchor=(0.5, 0.0))
  fig.tight_layout(rect=(0, 0.07, 1, 1))
  plt.show()

SIMBOLOS_PLOTLY = ["circle", "square", "diamond"]

def graficar_pca_3d(espacios):
  figura = make_subplots(rows=1, cols=len(espacios), specs=[[{"type": "scene"}] * len(espacios)], subplot_titles=list(espacios))
  for columna, (nombre, (vectores_train, vectores_val)) in enumerate(espacios.items(), start=1):
    proyeccion_train, proyeccion_val, varianza = proyectar_con_pca(vectores_train, vectores_val, 3)
    for clase in range(len(NOMBRES_CLASES)):
      for proyeccion, clases_del_grupo, grupo, tamano, opacidad in ((proyeccion_train, clases_train, "train", 2.5, 0.3), (proyeccion_val, clases_val, "validacion", 4, 1.0)):
        puntos = proyeccion[clases_del_grupo == clase]
        figura.add_trace(go.Scatter3d(
          x=puntos[:, 0], y=puntos[:, 1], z=puntos[:, 2], mode="markers", name=f"{NOMBRES_CLASES[clase]} ({grupo})",
          legendgroup=f"{NOMBRES_CLASES[clase]} ({grupo})", showlegend=(columna == 1),
          marker=dict(size=tamano, color=COLORES_CLASES[clase], symbol=SIMBOLOS_PLOTLY[clase], opacity=opacidad, line=dict(width=0)),
          hovertemplate=f"{NOMBRES_CLASES[clase]} ({grupo})<extra></extra>",
        ), row=1, col=columna)
    figura.update_scenes(xaxis_title=f"PC1 ({varianza[0]:.0%})", yaxis_title=f"PC2 ({varianza[1]:.0%})", zaxis_title=f"PC3 ({varianza[2]:.0%})", row=1, col=columna)

  # Sin width fijo, plotly usa todo el ancho de la celda
  figura.update_layout(template="simple_white", height=560, font=dict(color=COLOR_TINTA), legend=dict(orientation="h", y=-0.02), margin=dict(l=0, r=0, t=40, b=0))
  figura.show()

In [ ]:
graficar_pca_2d({"Tf-idf": (normalizar_filas(tfidf_train), normalizar_filas(tfidf_val))})

La versión 3D se puede **girar con el mouse**, hacer zoom con la rueda y ocultar una clase haciendo clic en la leyenda.

In [ ]:
graficar_pca_3d({"Tf-idf": (normalizar_filas(tfidf_train), normalizar_filas(tfidf_val))})

### **Para explorar**

> Cambiá `K` (50, 200, 800) y volvé a correr desde el k-means. ¿Cómo cambia la precisión? ¿Y el tiempo?
\
> Cambiá `IMAGENES_POR_CLASE` a 100. ¿Qué pasa con el vocabulario y con la búsqueda cuando hay pocas imágenes de train?
\
> En la búsqueda, ¿en qué consultas fallan los vecinos? ¿Qué tienen en común las imágenes que se confunden?

# **Parte 3: Autoencoder**

Hasta ahora describimos cada imagen con un vector de `K` números, uno por palabra visual. Ese vector tiene dos problemas:

> Es **ralo**: cada imagen usa unas pocas palabras de las `K`.
\
> Es **ruidoso**: dos imágenes parecidas pueden usar palabras visuales distintas pero parecidas entre sí (un borde un poco más inclinado cae en otra palabra). Para la similitud coseno, esas dos palabras no tienen nada en común.

Un **autoencoder** es un MLP que aprende a **reconstruir su propia entrada** pasando por un **cuello de botella**:

> **Encoder:** comprime el vector de `K` números en uno mucho más corto, el **embedding**.
\
> **Decoder:** intenta reconstruir el vector original a partir del embedding.

Como no entra todo por el cuello de botella, la red tiene que quedarse con lo que explica a la mayoría de las imágenes y descartar el detalle. Las palabras que suelen aparecer juntas terminan resumidas en una misma dirección del embedding. Después usamos ese embedding como nuevo vector para buscar.

En el entrenamiento **no usamos las clases**: la única señal es reconstruir bien el vector. Lo entrenamos con la loss de reconstrucción (error cuadrático medio):

$$
\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \lVert x_i - \hat{x}_i \rVert^2 \qquad \hat{x}_i = \text{decoder}(\text{encoder}(x_i))
$$

## **Imports**

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

## **Preparamos los vectores**

La entrada de la red son los vectores **tf-idf**. Antes los normalizamos por fila: una imagen con muchos keypoints tiene un vector más "largo" que una con pocos, y a la similitud coseno eso no le importa, pero a la loss de reconstrucción sí. Dejamos cada fila con norma $\sqrt{K}$, de modo que el valor cuadrático medio de cada entrada sea 1.

In [ ]:
def preparar_entrada(matriz_tfidf):
  return torch.tensor(np.sqrt(K) * normalizar_filas(matriz_tfidf), dtype=torch.float32)

entrada_train = preparar_entrada(tfidf_train)
entrada_val = preparar_entrada(tfidf_val)

# Como en la clase de MLP: un Dataset para acceder a los vectores y un DataLoader para iterar de a tandas.
# Aca no hay etiquetas: cada vector es su propio objetivo
class VectoresDataset(Dataset):
  def __init__(self, X):
    self.X = X

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx]


train_loader = DataLoader(VectoresDataset(entrada_train), batch_size=64, shuffle=True)

entrada_train.shape

## **El Autoencoder**

Es un MLP con dos mitades simétricas: `K` → 128 → `LATENTE` en el encoder y `LATENTE` → 128 → `K` en el decoder. `LATENTE` es el tamaño del cuello de botella: cuanto más chico, más obliga a resumir.

El parámetro `ruido` lo vamos a usar más adelante. Por ahora lo dejamos en 0.

In [ ]:
LATENTE = 16  #@param {type:"integer"}

class Autoencoder(nn.Module):
  def __init__(self, input_features, size_hidden=128, size_latente=16, ruido=0.0):
    super().__init__()
    self.ruido = nn.Dropout(ruido)
    self.encoder = nn.Sequential(nn.Linear(input_features, size_hidden), nn.ReLU(), nn.Linear(size_hidden, size_latente))
    self.decoder = nn.Sequential(nn.Linear(size_latente, size_hidden), nn.ReLU(), nn.Linear(size_hidden, input_features))

  # El ruido solo se aplica al entrenar: modelo.eval() apaga el Dropout
  def forward(self, x):
    return self.decoder(self.encoder(self.ruido(x)))


def calcular_embeddings(modelo, X):
  modelo.eval()
  with torch.no_grad():
    return modelo.encoder(X).numpy()

### Entrenamiento

El loop es el mismo de la clase de MLP: forward, loss, backward y un paso del optimizador. Las diferencias son que la "etiqueta" de cada vector es **él mismo**, y que ya no hay clases ni accuracy.

Cada 5 épocas también medimos la **precisión@5 de la búsqueda** sobre las imágenes de validación usando el embedding. Esa medida no entra en la loss: solo la miramos para ver si reconstruir mejor ayuda a buscar mejor.

In [ ]:
EPOCAS = 150  #@param {type:"integer"}
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CADA_CUANTAS_EPOCAS_MEDIR = 5

def medir_precision(modelo):
  return precision_en_k(calcular_embeddings(modelo, entrada_val), clases_val, calcular_embeddings(modelo, entrada_train), clases_train)

def entrenar_autoencoder(modelo, train_loader, epocas, learning_rate, weight_decay):
  criterio = nn.MSELoss()
  optimizer = torch.optim.Adam(modelo.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # La posicion 0 de cada lista es el modelo sin entrenar
  historial = {"loss_train": [], "loss_val": [], "epocas_precision": [], "precision_val": []}

  def registrar_epoca(epoca):
    modelo.eval()
    with torch.no_grad():
      historial["loss_train"].append(criterio(modelo(entrada_train), entrada_train).item())
      historial["loss_val"].append(criterio(modelo(entrada_val), entrada_val).item())
    if epoca % CADA_CUANTAS_EPOCAS_MEDIR == 0:
      historial["epocas_precision"].append(epoca)
      historial["precision_val"].append(medir_precision(modelo))

  registrar_epoca(0)

  for epoca in range(1, epocas + 1):
    modelo.train()
    for batch_X in train_loader:

      optimizer.zero_grad()

      reconstruccion = modelo(batch_X)

      # La reconstruccion se compara contra la propia entrada
      loss = criterio(reconstruccion, batch_X)

      # Calcular Gradientes
      loss.backward()

      # Actualizo los pesos
      optimizer.step()

    registrar_epoca(epoca)

    if epoca % 25 == 0:
      print(f"Epoch {epoca}, Train Loss: {historial['loss_train'][-1]:.4f}, Val Loss: {historial['loss_val'][-1]:.4f}, Precision@5 val: {historial['precision_val'][-1]:.3f}")

  return historial

In [ ]:
#@title Funcion para graficar las curvas { display-mode: "form" }

def graficar_curvas_autoencoder(historial, precision_de_referencia, titulo):
  epocas = np.arange(len(historial["loss_train"]))

  fig, (ax_loss, ax_precision) = plt.subplots(1, 2, figsize=(12, 4))

  ax_loss.plot(epocas, historial["loss_train"], color=COLOR_TINTA, linewidth=2, label="Train")
  ax_loss.plot(epocas, historial["loss_val"], color=COLOR_TINTA, linewidth=2, linestyle="--", alpha=0.55, label="Validacion")
  ax_loss.set_title("Loss de reconstruccion")
  ax_loss.set_xlabel("Epoca")
  ax_loss.legend(frameon=False)

  ax_precision.plot(historial["epocas_precision"], historial["precision_val"], color=COLOR_CLASE_1, linewidth=2, label="Embedding del autoencoder")
  ax_precision.axhline(precision_de_referencia, color=COLOR_TINTA, linestyle="--", linewidth=1.2, label="Tf-idf sin autoencoder")
  ax_precision.axhline(1 / len(NOMBRES_CLASES), color=COLOR_TINTA, linestyle=":", linewidth=1, label="Al azar")
  ax_precision.set_title("Precision@5 de la busqueda (validacion)")
  ax_precision.set_xlabel("Epoca")
  ax_precision.legend(frameon=False, loc="lower right")

  fig.suptitle(titulo, x=0.01, ha="left", fontsize=14, fontweight="bold")
  fig.tight_layout()
  plt.show()

#### Autoencoder simple

In [ ]:
precision_tfidf = precision_en_k(tfidf_val, clases_val, tfidf_train, clases_train)

torch.manual_seed(SEMILLA)
autoencoder_simple = Autoencoder(K, size_latente=LATENTE)

print(f"Parametros entrenables: {sum(parametro.numel() for parametro in autoencoder_simple.parameters())}")
historial_simple = entrenar_autoencoder(autoencoder_simple, train_loader, EPOCAS, LEARNING_RATE, WEIGHT_DECAY)

In [ ]:
graficar_curvas_autoencoder(historial_simple, precision_tfidf, "Autoencoder simple")

La precisión sube rápido en las primeras épocas y después se queda quieta, aunque la loss de train siga bajando. Y la loss de validación se despega de la de train: la red empieza a **memorizar** los vectores de entrenamiento. Reconstruir mejor no es lo mismo que buscar mejor: el autoencoder simple no tiene ningún motivo para que su embedding agrupe imágenes parecidas, solo para que pueda reconstruir el vector.

### Autoencoder con ruido (*denoising*)

Una forma de evitarlo: durante el entrenamiento **le rompemos la entrada** y le pedimos que reconstruya la versión original. Con `nn.Dropout` apagamos al azar una fracción `RUIDO` de las palabras de cada vector, y la loss compara contra el vector **completo**.

Para reconstruir una palabra que falta, la red no puede copiar la entrada: tiene que deducirla a partir de las otras. Eso la obliga a aprender qué palabras suelen aparecer juntas, que es justo lo que necesitamos.

In [ ]:
RUIDO = 0.3  #@param {type:"number"}

torch.manual_seed(SEMILLA)
autoencoder_con_ruido = Autoencoder(K, size_latente=LATENTE, ruido=RUIDO)

historial_con_ruido = entrenar_autoencoder(autoencoder_con_ruido, train_loader, EPOCAS, LEARNING_RATE, WEIGHT_DECAY)

In [ ]:
graficar_curvas_autoencoder(historial_con_ruido, precision_tfidf, f"Autoencoder con ruido ({RUIDO:.0%} de las palabras apagadas)")

Comparando con el autoencoder simple: la loss de reconstrucción de train queda más alta (reconstruir con ruido es más difícil), pero **la brecha con validación es menor** y la precisión de la búsqueda suele quedar más alta.

## **Centrar el embedding**

Un último detalle antes de comparar. La similitud coseno mide el **ángulo** entre dos vectores, y los embeddings de todas las imágenes suelen estar corridos hacia un mismo lado, hacia una "dirección promedio" compartida. Eso hace que todos los ángulos se parezcan y se pierda parte de lo que distingue a una imagen de otra.

La solución es restarle a todos los embeddings el **embedding promedio de train**. Usamos el promedio de train y no el de validación: las imágenes de validación son nuevas y no pueden influir en nada, ni siquiera en un promedio.

In [ ]:
def centrar_con_la_media_de_train(embeddings_val, embeddings_train):
  media_train = embeddings_train.mean(axis=0)
  return embeddings_val - media_train, embeddings_train - media_train

## **Comparando contra el resto**

Ponemos todo junto: la precisión@5 de cada representación sobre las imágenes de validación. Agregamos **PCA** con la misma cantidad de dimensiones que el embedding como referencia, porque es la forma "clásica" de comprimir un vector: es lo que aprendería un autoencoder sin activaciones no lineales. PCA ya resta el promedio por dentro, así que no hace falta centrarlo.

A la derecha, a quién eligen como vecino más cercano con el embedding centrado del autoencoder con ruido.

In [ ]:
pca_referencia = PCA(n_components=LATENTE, random_state=SEMILLA).fit(entrada_train.numpy())

embeddings_con_ruido_val = calcular_embeddings(autoencoder_con_ruido, entrada_val)
embeddings_con_ruido_train = calcular_embeddings(autoencoder_con_ruido, entrada_train)

representaciones = {
  "Tf-idf": (tfidf_val, tfidf_train),
  f"PCA ({LATENTE})": (pca_referencia.transform(entrada_val.numpy()), pca_referencia.transform(entrada_train.numpy())),
  "AE simple": (calcular_embeddings(autoencoder_simple, entrada_val), calcular_embeddings(autoencoder_simple, entrada_train)),
  "AE con ruido": (embeddings_con_ruido_val, embeddings_con_ruido_train),
  "AE con ruido centrado": centrar_con_la_media_de_train(embeddings_con_ruido_val, embeddings_con_ruido_train),
}

graficar_evaluacion(representaciones)

Lo que se ve:

> Los embeddings **mejoran la precisión respecto de tf-idf**, con apenas `LATENTE` números por imagen en lugar de `K`.
\
> El autoencoder con ruido suele ser el mejor, y el simple queda por debajo. **PCA se acerca bastante sin entrenar nada**: buena parte de la mejora viene de comprimir y descartar el ruido, no de que la red sea no lineal.
\
> **Centrar el embedding suma un poco más** (alrededor de un punto), sin volver a entrenar nada.
\
> La mejora es real pero moderada. Bag of Words no sabe dónde está cada keypoint (qué hay arriba o al lado de qué), y con eso solo no alcanza para separar bien tres clases tan variadas.

## **Cómo cambió el espacio**

Volvemos a proyectar con PCA, como en la parte 2. A la izquierda, los vectores **tf-idf** originales (`K` dimensiones). A la derecha, los **embeddings centrados** del autoencoder con ruido (`LATENTE` dimensiones). En los dos casos el PCA se ajusta solo con train, y los puntos grandes son imágenes de validación que la red nunca vio.

Lo que hay que mirar es si cada clase está más agrupada y separada de las otras que antes.

In [ ]:
consultas_embedding, base_embedding = representaciones["AE con ruido centrado"]
precision_embedding = precision_en_k(consultas_embedding, clases_val, base_embedding, clases_train)

espacios_antes_y_despues = {
  f"Tf-idf ({K} dim): precision@5 = {precision_tfidf:.2f}": (normalizar_filas(tfidf_train), normalizar_filas(tfidf_val)),
  f"Embedding ({LATENTE} dim): precision@5 = {precision_embedding:.2f}": (normalizar_filas(base_embedding), normalizar_filas(consultas_embedding)),
}

graficar_pca_2d(espacios_antes_y_despues)

In [ ]:
graficar_pca_3d(espacios_antes_y_despues)

Y la búsqueda de la parte 2, ahora con el embedding como una opción más. Probá las mismas consultas con **tf-idf** y con **AE con ruido centrado**.

In [ ]:
interact(
  mostrar_busqueda,
  indice_consulta=IntSlider(min=0, max=len(indices_val) - 1, step=1, value=0, description="Consulta", continuous_update=False),
  representacion=Dropdown(options=list(representaciones), value="AE con ruido centrado", description="Vectores"),
);

### **Para explorar**

> Cambiá `LATENTE` (2, 8, 16, 64) y volvé a entrenar. ¿Qué pasa con la precisión? ¿Y con la loss de reconstrucción?
> Cambiá `RUIDO` (0, 0.3, 0.6) y mirá cuánto cambia la precisión sin cambiar casi nada la loss.
> Probá con `IMAGENES_POR_CLASE = 100` en la parte 2. ¿El autoencoder sigue mejorando la búsqueda con tan pocos datos?